# Falcon One — simulação de voo com RocketPy


Este fluxo usa a biblioteca **RocketPy** e arquivos locais do modelo: curva do motor (`FalconOneV1.eng`), arrasto em função do Mach (`OnArrasto.csv` / `OffArrasto.csv`) e os parâmetros geométricos do foguete. A célula seguinte garante que o pacote esteja disponível no interpretador.

In [ ]:
#%pip install rocketpy

O que se segue monta, em sequência, o **ambiente** (coluna de ar e vento), o **motor Falcon One**, o **veículo** com geometria e arrasto, e por fim integra a **trajetória** (`Flight`). Os imports abaixo carregam as classes usadas nessa cadeia.


In [ ]:
from rocketpy import Environment, Flight, Rocket, SolidMotor

Saída em **SVG** para os gráficos gerados pelo matplotlib: melhor nitidez ao ampliar figuras nos relatórios.


In [ ]:
%config InlineBackend.figure_formats = ['svg']
%matplotlib inline

## Modelo para a simulação


### Ambiente no local de lançamento


O objeto `Environment` ancora o **local** (latitude, longitude, elevação do solo) e passa a prover, junto com o modelo atmosférico, a **densidade do ar**, **temperatura**, **velocidade do som** e **perfil de vento** em função da altitude — entradas diretas para arrasto, Mach e forças aerodinâmicas ao longo da trajetória simulada.

In [ ]:
# Configura as condições do local e data de lançamento
from datetime import datetime, timedelta
from rocketpy import Environment

tomorrow = datetime.now() + timedelta(days=1)

env = Environment(date=tomorrow)

env.set_location(latitude=-21.90795, longitude=-48.96156)

env.prints.launch_site_details()

env.set_atmospheric_model(type="forecast", file="GFS")

env.plots.atmospheric_model()

env.info()
#env = Environment(
#    latitude=-21.90795,          # Latitude do local de lançamento (em graus decimais)
#    longitude=-48.96156,         # Longitude do local de lançamento (em graus decimais)
#    elevation=495            # Altitude do local de lançamento (em metros)
#)

A **data e hora** em UTC escolhidas abaixo selecionam qual ciclo da previsão **GFS** alimenta a coluna atmosférica (no código, o dia é deslocado em relação à data corrente para fixar o cenário meteorológico usado na integração).


O modelo **Forecast / GFS** substitui uma atmosfera padrão estática por perfis horizontais consistentes com a previsão na época definida. Avisos do RocketPy sobre variáveis incompletas em algumas camadas significam apenas interpolação entre níveis do arquivo — não impedem o uso da simulação.


`env.info()` resume **gravidade local**, **local do lançamento**, **condições na superfície** e traça perfis de densidade, temperatura e vento — o mesmo pacote de dados que o integrador usará no `Flight`.


### Motor sólido Falcon One

O **`SolidMotor`** combina a curva de empuxo do arquivo **`FalconOneV1.eng`** com a geometria dos **três grãos**, **bocal** e massas/inércias secas. Durante a integração, o empuxo e a massa de propelente consumida alimentam as equações de movimento e o deslocamento do centro de massa do sistema.


In [ ]:
# %%
# ------------------------- MOTOR -------------------------
from rocketpy import SolidMotor

FalconOne = SolidMotor(
    # Fonte da curva de empuxo
    thrust_source="meteor-RASP_campeaon.eng",

    # Tempo de queima em segundos
    burn_time=1.4,

    # Massa do motor vazio (975.00 g -> 0.975 kg)
    dry_mass=0.975,

    # Inércia do motor vazio (convertida de g*mm² para kg*m²)(Ixx, Iyy, Izz)
    dry_inertia=(1.0047, 0.9756, 0.0298),

    # Centro de massa seco do motor no seu sistema (origem no bocal)
    center_of_dry_mass_position=0.10,

    # Posição da saída do bocal (origem do sistema do motor)
    nozzle_position=0.0,

    # Raio da SAÍDA do bocal (12.6 mm -> 0.0126 m)
    nozzle_radius=0.0126,

    # Raio da GARGANTA do bocal (4.9 mm -> 0.0049 m)
    throat_radius=0.0049,

    # Número de grãos de propelente
    grain_number=3,
    
    # Densidade do propelente em kg/m³
    grain_density=2050,
    
    # Raio externo do grão (21 mm -> 0.021 m)
    grain_outer_radius=0.021,
    
    # Raio interno inicial do grão (8.9 mm -> 0.0089 m)
    grain_initial_inner_radius=0.0089,
    
    # Altura/comprimento de um grão (80 mm -> 0.080 m)
    grain_initial_height=0.080,
    
    # Separação entre os grãos (0.5 mm -> 0.0005 m)
    grain_separation=0.0005,
    
    # Posição do centro de massa dos grãos (no sistema do motor)
    grains_center_of_mass_position=0.18,

    # Orientação do sistema de coordenadas do motor
    coordinate_system_orientation="nozzle_to_combustion_chamber"
)



Os comprimentos medidos ao longo do eixo do motor (**bocal, CM seco, CM dos grãos**) definem onde nasce o vetor de empuxo e como a inércia do conjunto motor–estrutura se acopla ao foguete. Referência: [posições no RocketPy](https://docs.rocketpy.org/en/latest/user/positions.html).


`FalconOne.info()` confronta o `.eng` com a geometria dos grãos: empuxo máximo, impulso total, massa de propelente e o gráfico **empuxo × tempo** — conferência antes de acoplar o motor ao veículo.


In [ ]:
# Imprime um resumo completo do motor para verificar se todos os dados foram carregados corretamente
FalconOne.info()


### Foguete e aerodinâmica


`FalconOneRocket` agrega **massa e inércia** da estrutura sem motor, **raio de referência** para arrasto, e duas curvas **Cd(Mach)** — motor ligado (`OnArrasto.csv`) e desligado (`OffArrasto.csv`) — que o integrador troca conforme o estado da queima. O referencial é **nariz → cauda**; o centro de massa sem motor está medido a partir da ponta da ogiva.


In [ ]:
# ------------------------- FOGUETE -------------------------
from rocketpy import Rocket

FalconOneRocket = Rocket(
    # Raio do foguete (diâmetro máx. 7,96 cm -> raio 0,0398 m)
    radius=0.0398,

    # Massa do foguete SEM o motor (2851 g -> 2.851 kg)
    mass=2.851,

    # Inércia do foguete (Ixx, Iyy, Izz) em kg*m^2
    inertia=(0.499, 0.499, 0.004),

    # Arquivo com a curva de arrasto por velocidade mach com motor desligado
    power_off_drag='OffArrasto.csv',

    # Arquivo com a curva de arrasto por velocidade mach com motor ligado
    power_on_drag='OnArrasto.csv',

    # Posição do centro de massa sem o motor (74 cm -> 0,74 m)
    # Medido a partir da ponta da ogiva (ponta do foguete)
    center_of_mass_without_motor=0.74,

    coordinate_system_orientation="nose_to_tail"
)

`add_motor` alinha o **FalconOne** ao eixo do foguete: o parâmetro de posição fixa a **saída do bocal** no referencial da ogiva, de modo que empuxo, arrasto e momentos atuem sobre a mesma linha de base estrutural.


In [ ]:

# A 'position' é a distância (em metros) da ponta da ogiva até a saída do bocal do motor.
FalconOneRocket.add_motor(FalconOne, position=1.41)

#### Superfícies e contribuição aerodinâmica


Ogiva em **série de potência**, **quatro aletas trapezoidais** e **transição de cauda** fecham a geometria externa: o RocketPy calcula a distribuição normal em função do Mach e da incidência efetiva, alimentando forças, momentos e a **margem estática** ao longo do voo. Todas as posições seguem o mesmo eixo **nariz–cauda** (ponta da ogiva na origem). [Referência de posicionamento](https://docs.rocketpy.org/en/latest/user/positions.html).

In [ ]:
# Adiciona a Ogiva ao foguete
ogiva = FalconOneRocket.add_nose(
    length=0.237,       # Comprimento da ogiva em metros
    kind='powerseries',  # Parabólica (OpenRocket)
    power=1.0,
    position=0        # A ponta da ogiva é a origem (0) do foguete
)

# Adiciona as Aletas ao foguete
aletas = FalconOneRocket.add_trapezoidal_fins(
    n=4,
    root_chord=0.08,
    tip_chord=0.03,
    span=0.115,
    position=1.26,
    cant_angle=0,
)

# Adiciona uma Cauda/Transição
cauda = FalconOneRocket.add_tail(
    top_radius=0.0398,   
    bottom_radius=0.025,  
    length=0.06,
    position=1.36
)


O desenho esquemático e o `info()` consolidam **dimensões**, **margem estática** inicial e coerência entre componentes — sanity check antes do `Flight`.


In [ ]:
# Verificando a representação do foguete
FalconOneRocket.plots.draw()
FalconOne.plots.draw()


In [ ]:
FalconOneRocket.all_info()

#### Recuperação por paraquedas


O paraquedas registrado abaixo usa **`Cd·S`**, **gatilho no apogeu** (máximo da trajetória integrada) e um modelo simples de **atraso de ejeção** e **ruído de altímetro**. Na descida, o integrador troca a dinâmica de voo livre por uma força de arrasto efetiva ligada a esse produto, reduzindo velocidade terminal antes do impacto.


In [ ]:
# %%
# ------------------------- SISTEMA DE RECUPERAÇÃO -------------------------

# Adiciona o Paraquedas Principal ao foguete
paraquedas_principal = FalconOneRocket.add_parachute(
    "ParaQuedas",       # Nome do paraquedas
    cd_s=2.3277,          # Vamos calcular juntos (Cd * Área)
    trigger="apogee",  # Acionamento no ponto mais alto do voo
    sampling_rate=100, # Taxa de amostragem do altímetro em Hz (ex: 150)
    lag=1.5,           # Atraso da ejeção em segundos (ex: 1.5)
    noise=(0, 0.1, 0.5)  # Ruído do altímetro (média, desvio_padrão, correlação)
)


Cada nova execução de `add_parachute` **acumula** outro sistema na mesma instância de `FalconOneRocket`; o `Flight` enxergaria desdobramentos e massas repetidas. Para iterar cenários, volte à montagem do foguete ou esvazie a lista antes de registrar de novo:

```python
FalconOneRocket.parachutes.clear()
```


## Integração da trajetória (`Flight`)

O objeto **`Flight`** resolve as equações de movimento acoplando **empuxo**, **arrasto**, **forças aerodinâmicas nas superfícies**, **vento em camadas** e a cinemática do **trilho** (`rail_length`, `inclination`, `heading`). A saída é o estado completo do veículo ao longo do tempo até o evento final — impacto ou descida sob paraquedas, conforme o modelo.


In [ ]:

test_flight = Flight(
    rocket=FalconOneRocket,
    environment=env,
    rail_length=4,
    inclination=85,
    heading=5,         
    max_time= 200,
    max_time_step=0.5,     # evita passos absurdamente pequenos
    min_time_step=0.5,
    verbose=True,          # imprime quando entra em cada fase
    time_overshoot=True,
    rtol = 0.5
)

## Séries temporais e diagnósticos

`all_info()` agrega **históricos** (altura, velocidade, Mach, ângulos, carga aerodinâmica, estabilidade, etc.) e visualizações agrupadas por tema: é o retrato pós-voo do mesmo modelo que acabou de ser integrado. Variáveis individuais permanecem acessíveis nos atributos do `Flight` para gráficos próprios.


In [ ]:
test_flight.all_info()

Exportação **KML**: a mesma trajetória integrada é projetada no globo (Google Earth ou equivalente), útil para inspecionar azimute, distância ao solo e forma da subida em relação ao campo de lançamento.


In [ ]:
test_flight.export_kml(
    file_name="Lançamento-LASC2025.kml",
    extrude=True,
    altitude_mode="relative_to_ground",
)

## Sensibilidades em torno do voo nominal

As rotinas abaixo **reaproveitam** o `Flight` já convergido como referência e recalculam grandezas-chave (**apogeu**, **velocidade na saída do trilho**, **estabilidade dinâmica**) quando massa, geometria ou condição de partida mudam — ou seja, leitura de projeto a partir do mesmo núcleo de simulação.


### Apogeu versus massa total

Varrendo a massa do veículo, o gráfico mostra como o **apogeu de referência** se desloca com carga útil ou margem de propelente: trade-off direto entre peso e altitude máxima para o mesmo motor e arrasto.


In [ ]:
from rocketpy.utilities import apogee_by_mass

apogee_by_mass(flight=test_flight, min_mass=5, max_mass=20, points=10, plot=True)

### Velocidade na saída do trilho versus massa

Na saída do trilho, o foguete já precisa de **velocidade suficiente** para o fluxo nas aletas ser estável (margem em relação ao **vento na superfície** da coluna GFS). O painel mostra essa velocidade limite em função da massa — região segura versus combinações pesadas ou vento forte.


In [ ]:
from rocketpy.utilities import liftoff_speed_by_mass

liftoff_speed_by_mass(flight=test_flight, min_mass=5, max_mass=20, points=10, plot=True)

### Estabilidade dinâmica versus envergadura

A **margem estática** num instante não descreve sozinha o amortecimento das oscilações. O experimento numérico abaixo **escala a envergadura das aletas**, reavalia a margem no trilho e no fim do voo, e calcula o **nível de estabilidade dinâmica** — ligação entre geometria, inércia e modo de curto período.


In [ ]:
# Classe auxiliar
import copy

from rocketpy import Function

# Preparar uma cópia do foguete
FalconOneRocket2 = copy.deepcopy(FalconOneRocket)

# Preparar a classe Environment
custom_env = Environment()
custom_env.set_atmospheric_model(type="custom_atmosphere", wind_v=-5)

# Simular diferentes margens estáticas variando a posição das aletas
simulation_results = []

for factor in [-0.5, -0.2, 0.1, 0.4, 0.7]:
    # Modificar conjunto de aletas removendo o anterior e adicionando um novo
    FalconOneRocket2.aerodynamic_surfaces.pop(-1)

    fin_set = FalconOneRocket2.add_trapezoidal_fins(
        n=4,
        root_chord=0.120,
        tip_chord=0.040,
        span=0.100,
        position=-1.04956 * factor,
    )
    # Simular
    print(
        "Simulando foguete com margem estática de {:1.3f}->{:1.3f} c".format(
            FalconOneRocket2.static_margin(0),
            FalconOneRocket2.static_margin(FalconOneRocket2.motor.burn_out_time),
        )
    )
    test_flight = Flight(
        rocket=FalconOneRocket2,
        environment=custom_env,
        rail_length=5.2,
        inclination=90,
        heading=0,
        max_time_step=0.01,
        max_time=5,
        terminate_on_apogee=True,
        verbose=True,
    )
    # Armazenar resultados
    static_margin_at_ignition = FalconOneRocket2.static_margin(0)
    static_margin_at_out_of_rail = FalconOneRocket2.static_margin(test_flight.out_of_rail_time)
    static_margin_at_steady_state = FalconOneRocket2.static_margin(test_flight.t_final)
    simulation_results += [
        (
            test_flight.attitude_angle,
            "{:1.2f} c | {:1.2f} c | {:1.2f} c".format(
                static_margin_at_ignition,
                static_margin_at_out_of_rail,
                static_margin_at_steady_state,
            ),
        )
    ]

Function.compare_plots(
    simulation_results,
    lower=0,
    upper=1.5,
    xlabel="Tempo (s)",
    ylabel="Ângulo de atitude (graus)",
)

### Frequência característica ao sair do trilho

Logo após a **saída do trilho**, o modo rígido–corpo que domina o pitch/yaw tem uma **frequência característica** associada à rigidez aerodinâmica e à inércia. O bloco abaixo extrai explicitamente esse espectro (complementar ao que já aparece em `all_info()`), com escolha fina das séries exibidas.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simular os primeiros 5 segundos de voo
flight = Flight(
    rocket=FalconOneRocket,
    environment=env,
    rail_length=5.2,
    inclination=90,
    heading=0,
    max_time_step=0.01,
    max_time=5,
)

# Realizar uma análise de Fourier
Fs = 100.0
# taxa de amostragem
Ts = 1.0 / Fs
# intervalo de amostragem
t = np.arange(1, 400, Ts)  # vetor de tempo
ff = 5
# frequência do sinal
y = flight.attitude_angle(t) - np.mean(flight.attitude_angle(t))
n = len(y)  # comprimento do sinal
k = np.arange(n)
T = n / Fs
frq = k / T  # faixa de frequência de dois lados
frq = frq[range(n // 2)]  # faixa de frequência de um lado
Y = np.fft.fft(y) / n  # cálculo e normalização da FFT
Y = Y[range(n // 2)]

# Criar o gráfico
fig, ax = plt.subplots(2, 1)
ax[0].plot(t, y)
ax[0].set_xlabel("Tempo")
ax[0].set_ylabel("Sinal")
ax[0].set_xlim((0, 5))
ax[0].grid()
ax[1].plot(frq, abs(Y), "r")  # plotando o espectro
ax[1].set_xlabel("Frequência (Hz)")
ax[1].set_ylabel("|Y(frequência)|")
ax[1].set_xlim((0, 5))
ax[1].grid()
plt.subplots_adjust(hspace=0.5)
plt.show()